# Version 2 — EEG-only collapse diagnostics

This notebook keeps the Version 1 CNN, binary dataset, and leave-one-participant-out protocol unchanged. It runs five fixed seeds and records the evidence needed to explain collapse: predictions, probabilities, confusion matrices, histories, best epochs, participant/channel statistics, distribution shifts, environment versions, Git revision, and data hashes.

A fold is collapsed only when it predicts one unique class. Results are written to Drive after every fold and completed folds are skipped when the notebook is resumed.

## 1. Setup

Use a GPU runtime. The notebook deliberately does not reinstall TensorFlow because Colab's TensorFlow build is matched to its CUDA runtime.

In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/parmisbathaeiyan/teco-eeg-nlp.git'
REPO_DIR = 'teco-eeg-nlp'

if not os.path.isdir('src'):
    if not os.path.isdir(REPO_DIR):
        subprocess.run(['git', 'clone', '-q', REPO_URL], check=True)
    os.chdir(REPO_DIR)

sys.path.insert(0, 'src')

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    pass

print('working directory:', os.getcwd())
print('Git commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

## 2. Paths and locked configuration

The configuration is committed with the code. Change paths here, but do not change seeds or hyperparameters after examining test results.

In [ ]:
import json

trt_dir = '/content/drive/MyDrive/Thesis/Data/TRT_Total'
labels_csv = '/content/drive/MyDrive/Thesis/Data/teco_sentiment_labels_task1.csv'
out_dir = '/content/drive/MyDrive/Thesis/Results/reproduce_yaser_codes/eegonly/v2_diagnoseCollapse'
v1_result_csv = '/content/drive/MyDrive/Thesis/Results/reproduce_yaser_codes/eegonly/v1/per_participant_eeg.csv'
config_path = 'configs/v2_diagnoseCollapse.json'

with open(config_path) as stream:
    config = json.load(stream)
config

## 3. Run or resume Version 2

A complete run contains 60 folds: 5 seeds × 12 held-out participants. Re-running this cell resumes from `fold_metrics.csv`.

In [ ]:
from v2_diagnoseCollapse import run_v2_diagnose_collapse

metrics = run_v2_diagnose_collapse(trt_dir, labels_csv, out_dir, config_path)

## 4. Overall and participant results

In [ ]:
import pandas as pd
from IPython.display import display

fold_metrics = pd.read_csv(os.path.join(out_dir, 'fold_metrics.csv'))
by_participant = pd.read_csv(os.path.join(out_dir, 'summary_by_participant.csv'))
by_seed = pd.read_csv(os.path.join(out_dir, 'summary_by_seed.csv'))

print('completed folds:', len(fold_metrics), '/ 60')
print('mean macro-F1:', round(fold_metrics.f1.mean(), 4))
print('fold-level F1 SD:', round(fold_metrics.f1.std(), 4))
print('collapses:', int(fold_metrics.collapsed.sum()), '/', len(fold_metrics))
display(by_participant.round(4))
display(by_seed.round(4))

## 5. Collapse-focused view

This table answers the first diagnostic question: do the same participants collapse repeatedly across seeds?

In [ ]:
collapse_grid = fold_metrics.pivot(index='participant', columns='seed', values='collapsed')
f1_grid = fold_metrics.pivot(index='participant', columns='seed', values='f1')

print('Collapse grid (True means one predicted class):')
display(collapse_grid)
print('Macro-F1 grid:')
display(f1_grid.round(3))

## 6. Participant attributes and signal statistics

Use this view to compare recurring collapse with demographics, source pickle group, missing/padded-word behavior, amplitude, dispersion, and zero rate.

In [ ]:
participant_meta = pd.read_csv(os.path.join(out_dir, 'participant_metadata.csv'))
feature_stats = pd.read_csv(os.path.join(out_dir, 'participant_feature_stats.csv'))
overall_stats = feature_stats.query("label == 'all'")

audit = (by_participant
         .merge(participant_meta, on='participant', how='left')
         .merge(overall_stats, on='participant', how='left'))
display(audit.sort_values(['collapse_rate', 'f1_mean'], ascending=[False, True]).round(4))

## 7. Compare with Version 1 and the thesis

In [ ]:
thesis_f1 = {
    'AMB': .66, 'ARM': .63, 'EIA': .69, 'ELD': .60,
    'FAS': .66, 'JAY': .76, 'MEB': .71, 'NAD': .72,
    'RAS': .66, 'SAS': .55, 'SHU': .64, 'AYB': .72,
}
comparison = by_participant.copy()
comparison['thesis_f1'] = comparison.participant.map(thesis_f1)
if os.path.exists(v1_result_csv):
    v1 = pd.read_csv(v1_result_csv)[['participant', 'f1']].rename(columns={'f1': 'v1_f1'})
    comparison = comparison.merge(v1, on='participant', how='left')
comparison['difference_from_thesis'] = comparison.f1_mean - comparison.thesis_f1
display(comparison.round(4))

## 8. Presentation-ready diagnostic figure

In [ ]:
import matplotlib.pyplot as plt

plot_data = by_participant.sort_values('f1_mean', ascending=False).reset_index(drop=True)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(plot_data.participant, plot_data.f1_mean,
            yerr=plot_data.f1_std.fillna(0), capsize=3)
axes[0].axhline(0.6666, color='tab:red', linestyle='--', label='Thesis mean')
axes[0].set_ylim(0, 1)
axes[0].set_ylabel('Macro-F1 across seeds')
axes[0].set_title('Version 2 performance and seed variability')
axes[0].legend()
axes[1].bar(plot_data.participant, plot_data.collapse_rate, color='tab:orange')
axes[1].set_ylim(0, 1)
axes[1].set_ylabel('Fraction of seeds collapsed')
axes[1].set_title('One-class collapse rate')
fig.tight_layout()
figures_dir = os.path.join(out_dir, 'figures')
os.makedirs(figures_dir, exist_ok=True)
fig.savefig(os.path.join(figures_dir, 'v2_diagnoseCollapse_participant_summary.png'), dpi=200, bbox_inches='tight')
plt.show()

## Output inventory

- `fold_metrics.csv`: one row per seed/participant fold, including confusion matrix and distribution-shift values.
- `summary_by_participant.csv`, `summary_by_seed.csv`, `summary_overall.json`: aggregate results.
- `predictions/`: sentence-level truth, prediction, and class probabilities.
- `histories/`: epoch-level loss, accuracy, and validation macro-F1.
- `participant_feature_stats.csv`, `participant_channel_stats.csv`, `participant_metadata.csv`: common-attribute evidence.
- `run_metadata.json`: configuration, environment, commit, device, and SHA-256 data fingerprints.